# Data cleaning - ABS Income (Data by Region) by SA2

This notebook cleans the ABS **Income (Data by Region)** dataset down to
**Victorian SA2 regions, all available years, two income measures**:

- median equivalised total household income (weekly).
- median total income (excl. Government pensions and
  allowances).

The cleaned table is saved as two CSV files - one with equivalised total household income for modelling rental prices and another with median total income for calculating affordability.

## Step 1 - Imports and configuration
Set the source file, the two output paths, the income column names and the
strings ABS uses to mark missing values. Keeping these in one place means the
rest of the notebook never hard-codes a column name or placeholder.

In [24]:
import re

import pandas as pd

FILE_PATH = "../../external_data/14100DO0004_2011-25.xlsx"
SHEET_NAME = "Table 1"
HEADER_ROW = 6

EQUIVALISED_OUTPUT_PATH = "../../external_data/abs_income_sa2_vic_clean.csv"
TOTAL_INCOME_OUTPUT_PATH = (
    "../../external_data/abs_total_income_sa2_vic_clean.csv"
)

EQUIVALISED_INCOME_COL = (
    "Median equivalised total household income (weekly) ($)"
)
TOTAL_INCOME_COL = (
    "Median total income (excl. Government pensions "
    "and allowances) ($)"
)
INCOME_COLS = [EQUIVALISED_INCOME_COL, TOTAL_INCOME_COL]

# "-" is suppressed/nil, "np" is not published, "nan" is a blank cell that
# became a string when the sheet was loaded as text.
NULL_PLACEHOLDERS = ["-", "np", "nan", ""]

## Step 2 - Load the sheet
Loaded as text so ABS placeholders (`-`, `np`) are not misread as numbers
before they are handled explicitly in Step 6.

In [25]:
raw_df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)
raw_df = raw_df.astype(str)
raw_df.head()

,Code,Label,Year,$1-$499 per week (%),$500-$999 per week (%),$1000-$1999 per week (%),$2000-$2999 per week (%),$3000 or more per week (%),Nil income (%),Negative income (%),...,Superannuation and annuity income as main source of income (%),Total income earners (excl. Government pensions and allowances) (no.),Total income earners (excl. Government pensions and allowances) - median age (years),Total income (excl. Government pensions and allowances) ($m),Median total income (excl. Government pensions and allowances) ($),Mean total income (excl. Government pensions and allowances) ($),Persons who reported gross capital gains (no.),Total value of gross capital gains ($m),Median value of gross capital gains ($),Mean value of gross capital gains ($)
0,AUS,Australia,2011.0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
1,AUS,Australia,2016.0,27.9,23.1,22.3,5.1,3.1,9,0.5,...,-,-,-,-,-,-,-,-,-,-
2,AUS,Australia,2017.0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,955098,50532.4,3654,52908
3,AUS,Australia,2018.0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,991906,53892.1,3773,54332
4,AUS,Australia,2019.0,-,-,-,-,-,-,-,...,1.5,15744685,41,998968.8,49096,63448,950769,45220.3,2922,47562


## Step 3 - Keep only the columns we need
Work on a copy so later edits never touch `raw_df`.

In [26]:
ID_COLS = ["Code", "Label", "Year"]

income_df = raw_df[[*ID_COLS, *INCOME_COLS]].copy()

## Step 4 - Filter to Victorian SA2 codes
`Code` mixes Australia / State / SA4 / SA3 / SA2 / GCCSA in one column. SA2
codes are exactly 9 digits, and Victoria's state prefix is `2`.

In [27]:
is_vic_sa2 = income_df["Code"].str.match(r"^2\d{8}$")

before_rows = len(income_df)
income_df = income_df[is_vic_sa2].copy()
print(f"{before_rows:,} rows -> {len(income_df):,} Victorian SA2 rows")

32,006 rows -> 5,742 Victorian SA2 rows


## Step 5 - Clean the year column
`Year` was read as a float, so after loading as text the values look like
`"2025.0"`. Convert it to a nullable integer and drop rows without a valid
year. No year is filtered out, so every available year is kept.

In [28]:
income_df["Year"] = pd.to_numeric(
    income_df["Year"], errors="coerce"
).astype("Int64")

before_rows = len(income_df)
income_df = income_df.dropna(subset=["Year"])
print(f"{before_rows:,} rows -> {len(income_df):,} rows with a valid year")

rows_per_year = income_df["Year"].value_counts().sort_index()
print(f"Years available: {rows_per_year.index.tolist()}")
rows_per_year

5,742 rows -> 5,742 rows with a valid year
Years available: [2011, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


Year
2011    522
2016    522
2017    522
2018    522
2019    522
2020    522
2021    522
2022    522
2023    522
2024    522
2025    522
Name: count, dtype: Int64

# Step 6 - Split into one table per income measure 
Each table has (`code`, `label`, `year` plus a single income column) so each measure is cleaned and null-filtered independently.

In [29]:
MEASURE_COLS = {
    "equivalised": EQUIVALISED_INCOME_COL,
    "total": TOTAL_INCOME_COL,
}

income_by_measure = {
    measure_name: income_df[[*ID_COLS, income_column]].copy()
    for measure_name, income_column in MEASURE_COLS.items()
}

## Step 7 - Convert placeholders to nulls, cast income to numbers, drop nulls
For each measure: mark placeholders as null, cast to float, report any
unexpected non-numeric value, then drop rows with a null income. Each measure
keeps every row that has a value for that measure.

In [30]:
def clean_income_measure(measure_df, income_column):
    """Convert placeholders to nulls, cast to float and drop null rows."""
    stripped_values = measure_df[income_column].str.strip()
    is_placeholder = (
        stripped_values.isin(NULL_PLACEHOLDERS) | stripped_values.isna()
    )
    numeric_values = pd.to_numeric(
        stripped_values.mask(is_placeholder), errors="coerce"
    )

    is_unexpected = numeric_values.isna() & ~is_placeholder
    print(f"Placeholder cells: {is_placeholder.sum():,}")
    print(
        "Unexpected non-numeric values (should be empty): "
        f"{stripped_values[is_unexpected].unique().tolist()}"
    )

    cleaned_df = measure_df.copy()
    cleaned_df[income_column] = numeric_values

    before_rows = len(cleaned_df)
    cleaned_df = cleaned_df.dropna(subset=[income_column])
    print(
        f"{before_rows:,} rows -> {len(cleaned_df):,} rows "
        f"({before_rows - len(cleaned_df):,} null-income rows dropped)"
    )
    return cleaned_df


for measure_name, income_column in MEASURE_COLS.items():
    print(f"--- {measure_name} ---")
    income_by_measure[measure_name] = clean_income_measure(
        income_by_measure[measure_name], income_column
    )

--- equivalised ---
Placeholder cells: 4,191
Unexpected non-numeric values (should be empty): []
5,742 rows -> 1,551 rows (4,191 null-income rows dropped)
--- total ---
Placeholder cells: 3,140
Unexpected non-numeric values (should be empty): []
5,742 rows -> 2,602 rows (3,140 null-income rows dropped)


## Step  8- Clean up column names
Make column names code-friendly. The trailing underscore left by `($)` is kept
on purpose so names match what downstream notebooks read. Sort each table by
SA2 code and year.

In [31]:
def clean_column_name(name):
    """Lowercase, drop punctuation, and replace spaces with underscores.

    Trailing punctuation such as "($)" leaves a trailing underscore. This is
    kept on purpose so the names match what downstream notebooks expect.
    """
    name = name.strip().lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name


for measure_name, measure_df in income_by_measure.items():
    measure_df.columns = [
        clean_column_name(column_name) for column_name in measure_df.columns
    ]
    income_by_measure[measure_name] = measure_df.sort_values(
        ["code", "year"]
    ).reset_index(drop=True)
    print(measure_name, income_by_measure[measure_name].columns.tolist())

equivalised ['code', 'label', 'year', 'median_equivalised_total_household_income_weekly_']
total ['code', 'label', 'year', 'median_total_income_excl_government_pensions_and_allowances_']


## Data quality checks
Ensure no nulls within data, no duplicates, check years of data available and ensure SA2 regions missing at least one year is 0.

In [32]:
for measure_name, measure_df in income_by_measure.items():
    print(f"--- {measure_name} ---")
    print(f"Rows: {len(measure_df):,}")
    print(f"Nulls (should be 0): {int(measure_df.isnull().sum().sum())}")

    duplicate_rows = measure_df[
        measure_df.duplicated(subset=["code", "year"], keep=False)
    ]
    print(f"Duplicate SA2-year rows: {len(duplicate_rows):,}")

    years_available = (
        measure_df["year"].drop_duplicates().sort_values().tolist()
    )
    years_per_sa2 = measure_df.groupby("code")["year"].nunique()
    print(f"Years available: {years_available}")

INCOME_CEILING = 5000

equivalised_df = income_by_measure["equivalised"]
equivalised_column = equivalised_df.columns[-1]
suspect_rows = equivalised_df[
    (equivalised_df[equivalised_column] < 0)
    | (equivalised_df[equivalised_column] > INCOME_CEILING)
]

--- equivalised ---
Rows: 1,551
Nulls (should be 0): 0
Duplicate SA2-year rows: 0
Years available: [2011, 2016, 2021]
--- total ---
Rows: 2,602
Nulls (should be 0): 0
Duplicate SA2-year rows: 0
Years available: [2019, 2020, 2021, 2022, 2023]


## Step 9 - Save the cleaned data
Export two CSV files without the index. Both have the same layout (`code`,
`label`, `year` plus a single income column) and the same rows; only the
income measure differs:

- equivalised total household income, for modelling rental prices
- median total income, for calculating affordability

In [33]:
OUTPUT_PATHS = {
    "equivalised": EQUIVALISED_OUTPUT_PATH,
    "total": TOTAL_INCOME_OUTPUT_PATH,
}

for measure_name, output_path in OUTPUT_PATHS.items():
    measure_df = income_by_measure[measure_name]
    measure_df.to_csv(output_path, index=False)
    print(f"Saved {len(measure_df):,} rows to {output_path}.")

Saved 1,551 rows to ../../external_data/abs_income_sa2_vic_clean.csv.
Saved 2,602 rows to ../../external_data/abs_total_income_sa2_vic_clean.csv.
